In [3]:
# ==========================================================
# Predictive Maintenance & Anomaly Detection
# Part 1: Data Loading, Cleaning & Feature Engineering
# ==========================================================

# Import Libraries
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

# ----------------------------------------------------------
# Load Dataset
# ----------------------------------------------------------

file_path = r"C:\Users\Lenovo\Downloads\Thales_Group_Manufacturing.csv"

df = pd.read_csv(file_path)

print("=" * 60)
print("DATASET LOADED SUCCESSFULLY")
print("=" * 60)

# ----------------------------------------------------------
# Dataset Overview
# ----------------------------------------------------------

print("\nFirst 5 Rows")
print(df.head())

print("\nShape")
print(df.shape)

print("\nColumns")
print(df.columns.tolist())

print("\nMissing Values")
print(df.isnull().sum())

print("\nDuplicate Rows")
print(df.duplicated().sum())

# ----------------------------------------------------------
# Remove Duplicates
# ----------------------------------------------------------

df.drop_duplicates(inplace=True)

# ----------------------------------------------------------
# Convert Date Column
# ----------------------------------------------------------

df["Date"] = pd.to_datetime(
    df["Date"],
    format="%d-%m-%Y"
)

# ----------------------------------------------------------
# Create DateTime Column
# ----------------------------------------------------------

df["DateTime"] = pd.to_datetime(
    df["Date"].dt.strftime("%Y-%m-%d") + " " + df["Timestamp"],
    errors="coerce"
)

# Remove invalid datetime rows

df = df.dropna(subset=["DateTime"])

# ----------------------------------------------------------
# Sort Data
# ----------------------------------------------------------

df = df.sort_values(
    by=["Machine_ID", "DateTime"]
)

df.reset_index(drop=True, inplace=True)

print("\nDate Processing Completed")

# ----------------------------------------------------------
# Encode Machine ID
# ----------------------------------------------------------

df["Machine_Code"] = (
    df["Machine_ID"]
    .astype("category")
    .cat.codes
)

# ----------------------------------------------------------
# Encode Operation Mode
# ----------------------------------------------------------

df["Operation_Mode_Code"] = (
    df["Operation_Mode"]
    .astype("category")
    .cat.codes
)

print("\nEncoding Completed")

# ----------------------------------------------------------
# Rolling Window Size
# ----------------------------------------------------------

window = 10

# ----------------------------------------------------------
# Rolling Temperature
# ----------------------------------------------------------

df["Temp_Rolling"] = (

    df.groupby("Machine_ID")["Temperature_C"]

    .transform(

        lambda x: x.rolling(
            window,
            min_periods=1
        ).mean()

    )

)

# ----------------------------------------------------------
# Rolling Vibration
# ----------------------------------------------------------

df["Vibration_Rolling"] = (

    df.groupby("Machine_ID")["Vibration_Hz"]

    .transform(

        lambda x: x.rolling(
            window,
            min_periods=1
        ).mean()

    )

)

# ----------------------------------------------------------
# Rolling Power
# ----------------------------------------------------------

df["Power_Rolling"] = (

    df.groupby("Machine_ID")["Power_Consumption_kW"]

    .transform(

        lambda x: x.rolling(
            window,
            min_periods=1
        ).mean()

    )

)

# ----------------------------------------------------------
# Rolling Error
# ----------------------------------------------------------

df["Error_Rolling"] = (

    df.groupby("Machine_ID")["Error_Rate_%"]

    .transform(

        lambda x: x.rolling(
            window,
            min_periods=1
        ).mean()

    )

)

print("\nRolling Features Created")

# ----------------------------------------------------------
# Sensor Deviations
# ----------------------------------------------------------

df["Temp_Deviation"] = (
    df["Temperature_C"]
    - df["Temp_Rolling"]
)

df["Vibration_Deviation"] = (
    df["Vibration_Hz"]
    - df["Vibration_Rolling"]
)

df["Power_Deviation"] = (
    df["Power_Consumption_kW"]
    - df["Power_Rolling"]
)

df["Error_Deviation"] = (
    df["Error_Rate_%"]
    - df["Error_Rolling"]
)

# ----------------------------------------------------------
# Vibration / Power Ratio
# ----------------------------------------------------------

df["Vibration_Power_Ratio"] = (

    df["Vibration_Hz"]

    /

    (df["Power_Consumption_kW"] + 1)

)

# ----------------------------------------------------------
# Maintenance Score Decay
# ----------------------------------------------------------

df["Maintenance_Decay"] = (

    100

    - df["Predictive_Maintenance_Score"]

)

# ----------------------------------------------------------
# Error Trend
# ----------------------------------------------------------

df["Error_Trend"] = (

    df.groupby("Machine_ID")["Error_Rate_%"]

    .diff()

)

df["Error_Trend"] = df["Error_Trend"].fillna(0)

# ----------------------------------------------------------
# Production Efficiency
# ----------------------------------------------------------

df["Production_Efficiency"] = (

    df["Production_Speed_units_per_hr"]

    /

    (df["Power_Consumption_kW"] + 1)

)

print("\nFeature Engineering Completed")

# ----------------------------------------------------------
# Baseline Behaviour
# ----------------------------------------------------------

baseline = (

    df.groupby("Machine_ID")

    .agg({

        "Temperature_C": "mean",

        "Vibration_Hz": "mean",

        "Power_Consumption_kW": "mean",

        "Error_Rate_%": "mean"

    })

)

print("\nBaseline Behaviour")

print(baseline.head())

# ----------------------------------------------------------
# Features for Scaling
# ----------------------------------------------------------

features = [

    "Temperature_C",

    "Vibration_Hz",

    "Power_Consumption_kW",

    "Network_Latency_ms",

    "Packet_Loss_%",

    "Quality_Control_Defect_Rate_%",

    "Production_Speed_units_per_hr",

    "Predictive_Maintenance_Score",

    "Error_Rate_%",

    "Temp_Deviation",

    "Vibration_Deviation",

    "Power_Deviation",

    "Error_Deviation",

    "Vibration_Power_Ratio",

    "Maintenance_Decay",

    "Error_Trend",

    "Production_Efficiency"

]

# ----------------------------------------------------------
# Standard Scaling
# ----------------------------------------------------------

scaler = StandardScaler()

df_scaled = df.copy()

df_scaled[features] = scaler.fit_transform(
    df_scaled[features]
)

print("\nFeature Scaling Completed")

# ----------------------------------------------------------
# Save Processed Dataset
# ----------------------------------------------------------

output_path = r"C:\Users\Lenovo\Downloads\processed_manufacturing.csv"

df_scaled.to_csv(
    output_path,
    index=False
)

print("\nProcessed Dataset Saved Successfully")

print(output_path)

# ----------------------------------------------------------
# Final Information
# ----------------------------------------------------------

print("\nFinal Dataset Shape")

print(df_scaled.shape)

print("\nFinal Columns")

print(df_scaled.columns.tolist())

print("\nFirst Five Rows")

print(df_scaled.head())

print("\nPART 1 COMPLETED SUCCESSFULLY")

DATASET LOADED SUCCESSFULLY

First 5 Rows
         Date Timestamp  Machine_ID Operation_Mode  Temperature_C  \
0  01-01-2025  00:00:00          39           Idle         74.138   
1  01-01-2025  00:01:00          29         Active         84.265   
2  01-01-2025  00:02:00          15         Active         44.280   
3  01-01-2025  00:03:00          43         Active         40.569   
4  01-01-2025  00:04:00           8           Idle         75.064   

   Vibration_Hz  Power_Consumption_kW  Network_Latency_ms  Packet_Loss_%  \
0         3.501                 8.612              10.651          0.208   
1         3.356                 2.269              29.112          2.228   
2         2.080                 6.144              18.357          1.639   
3         0.298                 4.068              29.154          1.161   
4         0.346                 6.226              34.029          4.797   

   Quality_Control_Defect_Rate_%  Production_Speed_units_per_hr  \
0                  

In [4]:
# ==========================================================
# Predictive Maintenance & Anomaly Detection
# Part 2 : Isolation Forest Model
# ==========================================================

import pandas as pd
import joblib

from sklearn.ensemble import IsolationForest

# ----------------------------------------------------------
# Load Processed Dataset
# ----------------------------------------------------------

file_path = r"C:\Users\Lenovo\Downloads\processed_manufacturing.csv"

df = pd.read_csv(file_path)

print("="*60)
print("Processed Dataset Loaded")
print("="*60)

# ----------------------------------------------------------
# Features Used For Model
# ----------------------------------------------------------

features = [

    "Temperature_C",

    "Vibration_Hz",

    "Power_Consumption_kW",

    "Network_Latency_ms",

    "Packet_Loss_%",

    "Quality_Control_Defect_Rate_%",

    "Production_Speed_units_per_hr",

    "Predictive_Maintenance_Score",

    "Error_Rate_%",

    "Temp_Deviation",

    "Vibration_Deviation",

    "Power_Deviation",

    "Error_Deviation",

    "Vibration_Power_Ratio",

    "Maintenance_Decay",

    "Error_Trend",

    "Production_Efficiency"

]

X = df[features]

# ----------------------------------------------------------
# Isolation Forest Model
# ----------------------------------------------------------

model = IsolationForest(

    n_estimators=200,

    contamination=0.05,

    random_state=42

)

model.fit(X)

print("\nIsolation Forest Trained Successfully")

# ----------------------------------------------------------
# Predictions
# ----------------------------------------------------------

df["Anomaly_Label"] = model.predict(X)

# -1 = anomaly
#  1 = normal

# ----------------------------------------------------------
# Anomaly Score
# ----------------------------------------------------------

scores = model.decision_function(X)

# Higher = More abnormal

df["Anomaly_Score"] = scores.max() - scores

# Normalize score between 0 and 1

df["Anomaly_Score"] = (

    (df["Anomaly_Score"] - df["Anomaly_Score"].min())

    /

    (

        df["Anomaly_Score"].max()

        -

        df["Anomaly_Score"].min()

    )

)

print("\nAnomaly Scores Generated")

# ----------------------------------------------------------
# Risk Classification
# ----------------------------------------------------------

def risk(score):

    if score >= 0.70:

        return "High"

    elif score >= 0.40:

        return "Medium"

    else:

        return "Low"

df["Maintenance_Risk"] = df["Anomaly_Score"].apply(risk)

print("\nRisk Classification Completed")

# ----------------------------------------------------------
# High Risk Machines
# ----------------------------------------------------------

high_risk = (

    df[df["Maintenance_Risk"]=="High"]

    ["Machine_ID"]

    .unique()

)

print("\nHigh Risk Machines")

print(high_risk)

# ----------------------------------------------------------
# KPI Calculations
# ----------------------------------------------------------

total_machines = df["Machine_ID"].nunique()

high_risk_count = (

    df[df["Maintenance_Risk"]=="High"]

    ["Machine_ID"]

    .nunique()

)

medium_risk_count = (

    df[df["Maintenance_Risk"]=="Medium"]

    ["Machine_ID"]

    .nunique()

)

low_risk_count = (

    df[df["Maintenance_Risk"]=="Low"]

    ["Machine_ID"]

    .nunique()

)

average_score = df["Anomaly_Score"].mean()

average_maintenance = df["Predictive_Maintenance_Score"].mean()

downtime_prevention_index = (

    high_risk_count /

    total_machines

) * 100

print("\n==============================")
print("KPI SUMMARY")
print("==============================")

print("Total Machines :", total_machines)

print("High Risk :", high_risk_count)

print("Medium Risk :", medium_risk_count)

print("Low Risk :", low_risk_count)

print("Average Anomaly Score :", round(average_score,3))

print("Average Maintenance Score :", round(average_maintenance,2))

print("Downtime Prevention Index :", round(downtime_prevention_index,2),"%")

# ----------------------------------------------------------
# Early Warning Lead Time
# ----------------------------------------------------------

lead_time = (

    df.groupby("Machine_ID")

    ["Anomaly_Score"]

    .mean()

)

print("\nAverage Machine Risk")

print(lead_time.head())

# ----------------------------------------------------------
# Save Model
# ----------------------------------------------------------

model_path = r"C:\Users\Lenovo\Downloads\isolation_forest.pkl"

joblib.dump(model, model_path)

print("\nModel Saved Successfully")

print(model_path)

# ----------------------------------------------------------
# Save Final Dataset
# ----------------------------------------------------------

output = r"C:\Users\Lenovo\Downloads\final_manufacturing.csv"

df.to_csv(output,index=False)

print("\nFinal Dataset Saved")

print(output)

# ----------------------------------------------------------
# Top 10 High Risk Machines
# ----------------------------------------------------------

top10 = (

    df.groupby("Machine_ID")["Anomaly_Score"]

    .mean()

    .sort_values(ascending=False)

    .head(10)

)

print("\nTop 10 High Risk Machines")

print(top10)

print("\nPart 2 Completed Successfully")

Processed Dataset Loaded

Isolation Forest Trained Successfully

Anomaly Scores Generated

Risk Classification Completed

High Risk Machines
[ 1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24
 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47 48
 49 50]

KPI SUMMARY
Total Machines : 50
High Risk : 50
Medium Risk : 50
Low Risk : 50
Average Anomaly Score : 0.367
Average Maintenance Score : 0.0
Downtime Prevention Index : 100.0 %

Average Machine Risk
Machine_ID
1    0.363973
2    0.369379
3    0.367160
4    0.366511
5    0.367227
Name: Anomaly_Score, dtype: float64

Model Saved Successfully
C:\Users\Lenovo\Downloads\isolation_forest.pkl

Final Dataset Saved
C:\Users\Lenovo\Downloads\final_manufacturing.csv

Top 10 High Risk Machines
Machine_ID
8     0.373128
34    0.372729
37    0.371945
42    0.371206
46    0.371150
24    0.370652
7     0.370163
9     0.370128
13    0.369728
27    0.369515
Name: Anomaly_Score, dtype: float64

Part 2 Completed Suc

In [7]:
# ==========================================================
# Predictive Maintenance Dashboard
# Part 3A : Dashboard Overview
# ==========================================================

import streamlit as st
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# ----------------------------------------------------------
# Page Configuration
# ----------------------------------------------------------

st.set_page_config(
    page_title="Predictive Maintenance Dashboard",
    page_icon="🏭",
    layout="wide"
)

st.title("🏭 Predictive Maintenance & Anomaly Detection")
st.markdown("### 6G-Integrated Smart Manufacturing Systems")

# ----------------------------------------------------------
# Load Dataset
# ----------------------------------------------------------

@st.cache_data
def load_data():
    file_path = r"C:\Users\Lenovo\Downloads\final_manufacturing.csv"
    df = pd.read_csv(file_path)

    df["Date"] = pd.to_datetime(df["Date"], dayfirst=True)
    df["DateTime"] = pd.to_datetime(df["DateTime"])

    return df

df = load_data()

# ----------------------------------------------------------
# Sidebar
# ----------------------------------------------------------

st.sidebar.title("Filters")

machine = st.sidebar.selectbox(
    "Select Machine",
    ["All"] + sorted(df["Machine_ID"].unique().tolist())
)

operation = st.sidebar.multiselect(
    "Operation Mode",
    options=sorted(df["Operation_Mode"].unique()),
    default=sorted(df["Operation_Mode"].unique())
)

risk = st.sidebar.multiselect(
    "Maintenance Risk",
    options=["Low","Medium","High"],
    default=["Low","Medium","High"]
)

date_range = st.sidebar.date_input(
    "Select Date Range",
    value=(df["Date"].min(), df["Date"].max())
)

# ----------------------------------------------------------
# Filter Data
# ----------------------------------------------------------

filtered = df.copy()

if machine != "All":
    filtered = filtered[
        filtered["Machine_ID"] == machine
    ]

filtered = filtered[
    filtered["Operation_Mode"].isin(operation)
]

filtered = filtered[
    filtered["Maintenance_Risk"].isin(risk)
]

if len(date_range) == 2:

    start_date = pd.to_datetime(date_range[0])
    end_date = pd.to_datetime(date_range[1])

    filtered = filtered[
        (filtered["Date"] >= start_date) &
        (filtered["Date"] <= end_date)
    ]

# ----------------------------------------------------------
# KPIs
# ----------------------------------------------------------

total_machines = filtered["Machine_ID"].nunique()

high_risk = filtered[
    filtered["Maintenance_Risk"]=="High"
]["Machine_ID"].nunique()

medium_risk = filtered[
    filtered["Maintenance_Risk"]=="Medium"
]["Machine_ID"].nunique()

low_risk = filtered[
    filtered["Maintenance_Risk"]=="Low"
]["Machine_ID"].nunique()

avg_score = filtered["Anomaly_Score"].mean()

avg_maintenance = filtered[
    "Predictive_Maintenance_Score"
].mean()

col1,col2,col3,col4,col5,col6 = st.columns(6)

col1.metric(
    "Machines",
    total_machines
)

col2.metric(
    "High Risk",
    high_risk
)

col3.metric(
    "Medium Risk",
    medium_risk
)

col4.metric(
    "Low Risk",
    low_risk
)

col5.metric(
    "Avg Anomaly",
    round(avg_score,3)
)

col6.metric(
    "Maintenance Score",
    round(avg_maintenance,2)
)

st.divider()

# ----------------------------------------------------------
# Risk Distribution
# ----------------------------------------------------------

left,right = st.columns(2)

risk_count = (
    filtered["Maintenance_Risk"]
    .value_counts()
    .reset_index()
)

risk_count.columns=[
    "Risk",
    "Count"
]

fig = px.pie(
    risk_count,
    names="Risk",
    values="Count",
    title="Maintenance Risk Distribution",
    hole=0.45
)

left.plotly_chart(
    fig,
    use_container_width=True
)

# ----------------------------------------------------------
# Operation Mode
# ----------------------------------------------------------

mode = (
    filtered["Operation_Mode"]
    .value_counts()
    .reset_index()
)

mode.columns=[
    "Mode",
    "Count"
]

fig = px.bar(
    mode,
    x="Mode",
    y="Count",
    title="Operation Mode Distribution"
)

right.plotly_chart(
    fig,
    use_container_width=True
)

st.divider()

# ----------------------------------------------------------
# Average Sensor Values
# ----------------------------------------------------------

sensor = filtered[
[
"Temperature_C",
"Vibration_Hz",
"Power_Consumption_kW",
"Network_Latency_ms",
"Packet_Loss_%"
]
].mean().reset_index()

sensor.columns=[
    "Sensor",
    "Average"
]

fig = px.bar(
    sensor,
    x="Sensor",
    y="Average",
    title="Average Sensor Readings"
)

st.plotly_chart(
    fig,
    use_container_width=True
)

# ----------------------------------------------------------
# Efficiency Status
# ----------------------------------------------------------

eff = (
    filtered["Efficiency_Status"]
    .value_counts()
    .reset_index()
)

eff.columns=[
    "Efficiency",
    "Count"
]

fig = px.bar(
    eff,
    x="Efficiency",
    y="Count",
    color="Efficiency",
    title="Efficiency Status"
)

st.plotly_chart(
    fig,
    use_container_width=True
)

st.divider()

# ----------------------------------------------------------
# Correlation Heatmap
# ----------------------------------------------------------

corr = filtered[
[
"Temperature_C",
"Vibration_Hz",
"Power_Consumption_kW",
"Network_Latency_ms",
"Packet_Loss_%",
"Predictive_Maintenance_Score",
"Error_Rate_%",
"Anomaly_Score"
]
].corr()

fig = px.imshow(
    corr,
    text_auto=True,
    title="Correlation Matrix"
)

st.plotly_chart(
    fig,
    use_container_width=True
)

st.divider()

st.subheader("Dataset Preview")

st.dataframe(
    filtered.head(20),
    use_container_width=True
)

2026-07-14 14:24:50.835 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-14 14:24:50.837 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-14 14:24:50.945 
  command:

    streamlit run C:\Users\Lenovo\AppData\Local\Programs\Python\Python313\Lib\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-07-14 14:24:50.946 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-14 14:24:50.947 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-14 14:24:50.949 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-14 14:24:50.950 Thread 'MainThread': missing ScriptRunContext! This w

ValueError: unconverted data remains when parsing with format "%Y-%d-%m": "3", at position 12. You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.